# OULAD learning journeys with FastPath

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jose-alvarado-guzman/oulad/blob/main/notebooks/aga_fastpath_journeys.ipynb)

`aga_student_cohorts.ipynb` compares students on **which** materials they touched — a set.
This one compares them on the **shape of the journey**: what they did, in what order, how
long ago. FastPath is built for exactly this kind of sequence — clickstreams, customer
journeys, event logs — and OULAD's VLE data is a clickstream.

The catch is that the loaded graph has no sequence in it. `REVIEWED_MATERIAL` is a direct
student→material edge carrying a date, with nothing linking one interaction to the next. So
this notebook **builds an event chain first**:

```
(:Student)-[:FIRST_INTERACTION]->(:Interaction)-[:NEXT_INTERACTION]->(:Interaction)-> ...
                                       |
                                 [:OF_MATERIAL]-> (:EducationalMaterial)
```

One `:Interaction` per (student, material, day), chained in date order. Then FastPath turns
each student's chain into an embedding, KNN finds students with similar trajectories, and
Louvain groups them.

## Before you start

Same secrets as the cohorts notebook: `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD` and
`AURA_CLIENT_ID`, `AURA_CLIENT_SECRET`, `AURA_PROJECT_ID`, with *Notebook access* on. Run
[`oulad_data_load.ipynb`](oulad_data_load.ipynb) first if the graph is not loaded.

> **This notebook writes to your database.** Step 5 creates roughly 281,000 `:Interaction`
> nodes and their relationships for the default module. That is a much larger footprint than
> the cohorts notebook, which only sets a property. **Step 14 deletes all of it**, and step 5
> is skippable if a chain is already there. Nothing else is modified — the OULAD graph itself
> is only read.

> **A session is billed compute**, separate from AuraDB and independent of the write above.
> Step 12 deletes it; the TTL in step 8 is only a backstop.

## FastPath is in preview

There are no Aura-native docs for it yet. The closest public reference is the Snowflake Graph
Analytics documentation, which covers the same algorithm:
<https://neo4j.com/docs/snowflake-graph-analytics/current/algorithms/fastpath/>

Its Python surface is also still moving. Every parameter below was read off
`graphdatascience==2.0a5` directly, and several were renamed from earlier alphas
(`dimension` → `embedding_dimension`, `max_elapsed_time` → `lookback_horizon`,
`num_elapsed_times` → `num_time_anchors`, `time_node_property` →
`event_node_time_property`, `output_time` → `observation_time`, `decay_factor` →
`decay_rate`). Examples written against 2.0a1 will not run as-is.

## 1. Setup

Re-running resets the checkout to `origin/main`, discarding local changes.

In [ ]:
import os
import subprocess
import sys

REPO_URL = 'https://github.com/jose-alvarado-guzman/oulad.git'
REPO_DIR = '/content/oulad'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def run(*command):
    result = subprocess.run(command, text=True, capture_output=True)
    print((result.stdout + result.stderr).strip())
    result.check_returncode()

if IN_COLAB:
    if os.path.isdir(os.path.join(REPO_DIR, '.git')):
        run('git', '-C', REPO_DIR, 'fetch', '--depth', '1', 'origin', 'main')
        run('git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main')
        run('git', '-C', REPO_DIR, 'clean', '-fd')
    else:
        run('git', 'clone', '--depth', '1', REPO_URL, REPO_DIR)
    run('git', '-C', REPO_DIR, 'log', '-1', '--format=%h %ad %s', '--date=short')
    print()
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r',
         os.path.join(REPO_DIR, 'requirements-aga.txt')], check=True)
    print('Dependencies installed.')
else:
    REPO_DIR = os.getcwd()
    while REPO_DIR != '/' and not os.path.isdir(os.path.join(REPO_DIR, '.git')):
        REPO_DIR = os.path.dirname(REPO_DIR)
    print('Local kernel; assuming requirements-aga.txt is installed.')
    print('Repository root:', REPO_DIR)

## 2. Imports

In [ ]:
import os
import sys
from datetime import timedelta

REPO_DIR = '/content/oulad' if os.path.isdir('/content/oulad') else REPO_DIR
SRC_DIR = os.path.join(REPO_DIR, 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

for name in [m for m in sys.modules if m == 'oulad' or m.startswith('oulad.')]:
    del sys.modules[name]

import matplotlib.pyplot as plt
import pandas as pd
from neo4j import GraphDatabase
from graphdatascience.session import (
    AlgorithmCategory, AuraAPICredentials, DbmsConnectionInfo, GdsSessions)

import graphdatascience
from oulad.credentials import (
    AGA_SECRETS, ETL_SECRETS, MissingCredentialsError, aura_instance_id, load_credentials)
from oulad.logger import get_logger

print('graphdatascience', graphdatascience.__version__)
print('repository      ', REPO_DIR)

## 3. Credentials and the database connection

In [ ]:
logger = get_logger(REPO_DIR)

try:
    print('resolved from:', load_credentials(logger, required=ETL_SECRETS + AGA_SECRETS))
except MissingCredentialsError as error:
    raise SystemExit(f'\n{error}\n\nAdd the missing secrets in the sidebar, switch on '
                     'Notebook access, then re-run this cell.')

NEO4J_URI = os.environ['NEO4J_URI']
NEO4J_USERNAME = os.environ['NEO4J_USERNAME']
NEO4J_PASSWORD = os.environ['NEO4J_PASSWORD']
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE') or None
AURA_INSTANCE_ID = aura_instance_id(logger)

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()
print('connected to AuraDB, instance', AURA_INSTANCE_ID)

sessions = GdsSessions(api_credentials=AuraAPICredentials(
    os.environ['AURA_CLIENT_ID'], os.environ['AURA_CLIENT_SECRET'],
    os.environ['AURA_PROJECT_ID']))

## 4. Choose the scope and read the time span

One module at a time. The chain is one node per interaction, so the module size *is* the
write size — and every event has to fit inside FastPath's lookback window, which the dates
below determine.

| Module | Students | Events | Avg chain |
| --- | --- | --- | --- |
| `AAA` | 702 | 280,990 | 400 |
| `GGG` | 2,359 | 281,277 | 119 |
| `EEE` | 2,634 | 758,220 | 288 |
| `CCC` | 3,852 | 884,889 | 230 |
| `BBB` | 6,484 | 1,139,085 | 176 |
| `DDD` | 5,407 | 1,866,156 | 345 |
| `FFF` | 6,799 | 3,248,703 | 478 |

`GGG` is the default: enough students for the grouping to mean something, the smallest write,
and short chains. `MAX_STUDENTS` caps the build while you are trying things out — set it to
`None` for the whole module.

OULAD dates are days relative to the module start and can be negative (material viewed before
the course opened), so they are shifted to begin at 0.

In [ ]:
MODULE = 'GGG'
MAX_STUDENTS = None        # e.g. 200 for a quick trial, None for the whole module

SPAN_QUERY = '''
MATCH (s:Student)-[r:REVIEWED_MATERIAL]->(m:EducationalMaterial)<-[:HAS_MATERIAL]-(c:Course)
WHERE c.codeModule = $module
RETURN count(DISTINCT s) AS students, count(r) AS events,
       min(r.date) AS minDate, max(r.date) AS maxDate
'''
TYPES_QUERY = '''
MATCH (m:EducationalMaterial)<-[:HAS_MATERIAL]-(c:Course)
WHERE c.codeModule = $module
RETURN DISTINCT m.activityType AS activityType ORDER BY activityType
'''

with driver.session(database=NEO4J_DATABASE) as session:
    span = session.run(SPAN_QUERY, module=MODULE).single()
    activity_types = [r['activityType']
                      for r in session.run(TYPES_QUERY, module=MODULE)]

# GDS node properties must be numeric, so the activity type is encoded as an
# integer. FastPath's event_node_categorical_properties expects that too --
# hence event_node_ignored_category being an int in its signature.
TYPE_IDS = {name: index for index, name in enumerate(activity_types)}

SHIFT = -min(0, span['minDate'])          # move day 0 to the earliest event
OBSERVATION_TIME = float(span['maxDate'] + SHIFT + 1)
LOOKBACK_HORIZON = int(OBSERVATION_TIME) + 10   # must exceed the oldest elapsed time
NUM_TIME_ANCHORS = 20

print(f"module {MODULE}: {span['students']:,} students, {span['events']:,} events")
print(f"dates {span['minDate']} to {span['maxDate']}, shifted by +{SHIFT} "
      f'-> 0 to {span["maxDate"] + SHIFT}')
print(f'observation_time {OBSERVATION_TIME:.0f}, lookback_horizon {LOOKBACK_HORIZON}, '
      f'{NUM_TIME_ANCHORS} time anchors '
      f'({LOOKBACK_HORIZON / NUM_TIME_ANCHORS:.1f} days per anchor)')
print(f'\nactivity types encoded: {TYPE_IDS}')

## 5. Build the event chain

**This writes to your database.** One `:Interaction` per (student, material, day) — verified
unique in the source, so no aggregation is needed — chained in date order with ties broken by
material id so the sequence is deterministic.

Batched by student, because one transaction for a whole module is a bad idea. Re-running is
safe: students that already have a chain are skipped, so an interrupted build resumes.

Each `:Interaction` carries `day` (shifted), `clicks`, `activityTypeId`, `seq`, and `module`.
The `module` tag is what makes the scope and the cleanup in step 14 precise.

In [ ]:
BATCH = 100

CANDIDATES_QUERY = '''
MATCH (s:Student)-[:REVIEWED_MATERIAL]->(:EducationalMaterial)<-[:HAS_MATERIAL]-(c:Course)
WHERE c.codeModule = $module AND NOT (s)-[:FIRST_INTERACTION]->(:Interaction)
RETURN DISTINCT s.id AS studentId ORDER BY studentId
'''

BUILD_QUERY = '''
UNWIND $studentIds AS studentId
MATCH (s:Student {id: studentId})-[r:REVIEWED_MATERIAL]->(m:EducationalMaterial)
      <-[:HAS_MATERIAL]-(c:Course)
WHERE c.codeModule = $module
WITH s, m, r ORDER BY r.date, m.id
WITH s, collect({material: m, day: r.date + $shift, clicks: r.sumClick,
                 typeId: $typeIds[m.activityType]}) AS events
UNWIND range(0, size(events) - 1) AS i
WITH s, i, events[i] AS event
// the material has to be bound to its own variable: a node pulled out of a map
// cannot be used directly inside a CREATE pattern
WITH s, i, event.material AS material, event.day AS day,
     event.clicks AS clicks, event.typeId AS typeId
CREATE (ev:Interaction {module: $module, studentId: s.id, seq: i, day: day,
                        clicks: clicks, activityTypeId: typeId,
                        // FastPath reads numeric event features as a vector, so the
                        // click count has to be a list even though it is one number.
                        // Logged: raw totals are heavily skewed.
                        features: [log(toFloat(clicks) + 1.0)]})
CREATE (ev)-[:OF_MATERIAL]->(material)
WITH s, ev ORDER BY ev.seq
WITH s, collect(ev) AS chain
// list elements need binding too, for the same reason as the material above
WITH s, chain, chain[0] AS firstEvent
CREATE (s)-[:FIRST_INTERACTION]->(firstEvent)
WITH chain
UNWIND range(0, size(chain) - 2) AS j
WITH chain[j] AS previous, chain[j + 1] AS following
CREATE (previous)-[:NEXT_INTERACTION]->(following)
RETURN count(*) AS links
'''

with driver.session(database=NEO4J_DATABASE) as session:
    pending = [r['studentId'] for r in session.run(CANDIDATES_QUERY, module=MODULE)]

if MAX_STUDENTS is not None:
    pending = pending[:MAX_STUDENTS]

print(f'{len(pending):,} students still need a chain')
built = 0
for start in range(0, len(pending), BATCH):
    chunk = pending[start:start + BATCH]
    with driver.session(database=NEO4J_DATABASE) as session:
        session.run(BUILD_QUERY, studentIds=chunk, module=MODULE,
                    shift=SHIFT, typeIds=TYPE_IDS).consume()
    built += len(chunk)
    if built % (BATCH * 5) == 0 or built == len(pending):
        print(f'  {built:,}/{len(pending):,} students', flush=True)
print('chain built' if pending else 'nothing to do, chain already present')

## 6. Check the chain before trusting it

Three things worth confirming, because a broken chain would still embed — just wrongly:

- one `FIRST_INTERACTION` per student with a chain, and no student with two,
- as many `NEXT_INTERACTION` links as events minus students, since each chain of length *n*
  contributes *n-1* links,
- days along a chain never decrease.

In [ ]:
CHECK_QUERY = '''
MATCH (i:Interaction {module: $module})
WITH count(i) AS events
MATCH (:Student)-[f:FIRST_INTERACTION]->(:Interaction {module: $module})
WITH events, count(f) AS firsts
MATCH (:Interaction {module: $module})-[n:NEXT_INTERACTION]->()
RETURN events, firsts, count(n) AS nexts
'''
ORDER_QUERY = '''
MATCH (a:Interaction {module: $module})-[:NEXT_INTERACTION]->(b:Interaction)
WHERE b.day < a.day
RETURN count(*) AS outOfOrder
'''
with driver.session(database=NEO4J_DATABASE) as session:
    counts = session.run(CHECK_QUERY, module=MODULE).single()
    out_of_order = session.run(ORDER_QUERY, module=MODULE).single()['outOfOrder']

expected_nexts = counts['events'] - counts['firsts']
print(f"events {counts['events']:,}, chains {counts['firsts']:,}, "
      f"next links {counts['nexts']:,} (expected {expected_nexts:,})")
print('link count consistent:', counts['nexts'] == expected_nexts)
print('interactions out of date order:', out_of_order)
if counts['nexts'] != expected_nexts or out_of_order:
    raise SystemExit('The chain is not well formed; embedding it would be meaningless. '
                     'Delete it with step 14 and rebuild.')
print('\nchain looks sound')

## 7. A journey, in the raw

Worth seeing what FastPath is being handed before it turns into 128 numbers.

In [ ]:
SAMPLE_QUERY = '''
MATCH (s:Student)-[:FIRST_INTERACTION]->(first:Interaction {module: $module})
WITH s, first ORDER BY s.id LIMIT 1
MATCH path = (first)-[:NEXT_INTERACTION*0..14]->(ev:Interaction)
WITH s, ev ORDER BY ev.seq LIMIT 15
MATCH (ev)-[:OF_MATERIAL]->(m:EducationalMaterial)
RETURN s.id AS studentId, ev.seq AS seq, ev.day AS day,
       m.activityType AS activityType, ev.activityTypeId AS typeId, ev.clicks AS clicks
ORDER BY seq
'''
with driver.session(database=NEO4J_DATABASE) as session:
    sample = pd.DataFrame(session.run(SAMPLE_QUERY, module=MODULE).data())
print(f'first 15 events of one student\'s journey')
print(sample.to_string(index=False))

## 8. Size and open the session

In [ ]:
COUNT_QUERY = '''
MATCH (i:Interaction {module: $module})
WITH count(i) AS events
MATCH (s:Student)-[:FIRST_INTERACTION]->(:Interaction {module: $module})
WITH events, count(DISTINCT s) AS students
MATCH (:Interaction {module: $module})-[n:NEXT_INTERACTION]->()
RETURN events, students, events + count(n) AS relationships
'''
with driver.session(database=NEO4J_DATABASE) as session:
    sized = session.run(COUNT_QUERY, module=MODULE).single()

node_count = sized['events'] + sized['students']
relationship_count = sized['relationships']
print(f"{sized['students']:,} students + {sized['events']:,} interactions "
      f'= {node_count:,} nodes, {relationship_count:,} relationships')

memory = sessions.estimate(
    node_count=node_count,
    relationship_count=relationship_count,
    algorithm_categories=[AlgorithmCategory.NODE_EMBEDDING,
                          AlgorithmCategory.SIMILARITY,
                          AlgorithmCategory.COMMUNITY_DETECTION],
    node_label_count=2,          # Student, Interaction
    node_property_count=4,       # id, day, clicks, activityTypeId
)
print('estimated memory:', memory)

SESSION_NAME = f"oulad-fastpath-{os.environ['AURA_CLIENT_ID'][:8]}"
gds = sessions.get_or_create(
    session_name=SESSION_NAME,
    memory=memory,
    db_connection=DbmsConnectionInfo(
        aura_instance_id=AURA_INSTANCE_ID,
        username=NEO4J_USERNAME, password=NEO4J_PASSWORD, database=NEO4J_DATABASE),
    ttl=timedelta(hours=2),
)
print('session ready:', SESSION_NAME)

## 9. Project the chain

Both chain relationship types in one query. The source of a row is a `Student` for
`FIRST_INTERACTION` and an `Interaction` for `NEXT_INTERACTION`, so the property map asks for
the union of both — a node simply has no value for the keys that do not apply to it.

Everything FastPath reads has to be numeric, which is why `activityTypeId` is projected rather
than `activityType`.

In [ ]:
GRAPH_NAME = 'oulad-journeys'

PROJECTION_QUERY = '''
MATCH (src)-[r:FIRST_INTERACTION|NEXT_INTERACTION]->(tgt:Interaction)
WHERE tgt.module = $module
RETURN gds.graph.project.remote(src, tgt, {
    sourceNodeLabels: labels(src),
    targetNodeLabels: labels(tgt),
    sourceNodeProperties: src { .id, .day, .clicks, .activityTypeId, .features },
    targetNodeProperties: tgt { .day, .clicks, .activityTypeId, .features },
    relationshipType: type(r)
})
'''

gds.graph.project.cypher(
    graph_name=GRAPH_NAME, query=PROJECTION_QUERY,
    query_parameters={'module': MODULE}, overwrite=True)
G = gds.graph.get(GRAPH_NAME)
print(f'projected {G.node_count():,} nodes and {G.relationship_count():,} relationships')
print('node properties        :', G.node_properties())
print('relationship properties:', G.relationship_properties())

for label, needed in [('Interaction', {'day', 'activityTypeId', 'features'})]:
    missing = needed - set(G.node_properties().get(label, []))
    if missing:
        raise SystemExit(f'{label} is missing {missing} in the projection; FastPath would '
                         'either fail or read defaults. Check the projection query.')
print('\nthe properties FastPath needs are present')

## 10. FastPath embeddings

Each student's chain becomes one vector. The parameters worth understanding:

- **`observation_time`** is the vantage point. Elapsed time is measured back from here, and
  events at or after it are excluded — so it is set past the last event.
- **`lookback_horizon`** is how far back to look; it must exceed the oldest elapsed time or the
  earliest events fall outside the window and are dropped.
- **`num_time_anchors`** buckets that window. Twenty anchors over ~296 days is a fortnight
  each: enough to tell "worked steadily" from "crammed at the end".
- **`event_node_categorical_properties`** is what the events are *made of* — the kind of
  material touched.
- **`event_node_feature_vector_property`** is *how much* each event was worth. This carries
  the logged click count.
- **`random_seed`** is fixed so the embedding is reproducible.

> The feature vector was missing from the first version of this notebook, and its absence
> mattered. `sumClick` was copied onto the event nodes and projected into the session, then
> never passed to the algorithm — so a page opened once and a page hammered fifty times were
> the same event, and the embedding only ever saw *what kind of thing, when*. The run made
> without it produced 11 cohorts that were near-identical on every interpretable axis
> (mean day 109–119 across all of them) and separated outcomes by about 10 percentage points,
> against roughly 90 for click volume alone. Any comparison against that earlier run should
> treat it as a different configuration, not a verdict on FastPath.

In [ ]:
EMBEDDING_PROPERTY = 'journeyEmbedding'

embedding = gds.fast_path.mutate(
    G,
    base_node_label='Student',
    event_node_label='Interaction',
    mutate_property=EMBEDDING_PROPERTY,
    embedding_dimension=128,
    lookback_horizon=LOOKBACK_HORIZON,
    num_time_anchors=NUM_TIME_ANCHORS,
    event_node_categorical_properties=['activityTypeId'],
    event_node_feature_vector_property='features',   # click intensity
    event_node_time_property='day',
    first_relationship_type='FIRST_INTERACTION',
    next_relationship_type='NEXT_INTERACTION',
    observation_time=OBSERVATION_TIME,
    smoothing_window=2,
    smoothing_rate=10.0 / LOOKBACK_HORIZON,
    random_seed=42,
)
print(embedding)

## 11. Similar journeys, and the groups they form

KNN over the embeddings rather than node similarity: there are no shared neighbours to count
here, only vectors to compare. Louvain then groups students whose trajectories resemble each
other, and the cross-tab asks the same question as the cohorts notebook — except these groups
are built from *sequence*, not from set overlap.

In [ ]:
def as_frame(streamed, name):
    """Normalise a single-property stream to columns nodeId and `name`."""
    value_column = [c for c in streamed.columns if c != 'nodeId'][-1]
    return streamed.rename(columns={value_column: name})[['nodeId', name]]

knn = gds.knn.mutate(
    G,
    mutate_relationship_type='SIMILAR_JOURNEY',
    mutate_property='similarity',
    node_labels=['Student'],
    node_properties=EMBEDDING_PROPERTY,
    top_k=10,
)
print(f'{knn.relationships_written:,} SIMILAR_JOURNEY relationships')

louvain = gds.louvain.mutate(
    G, mutate_property='journeyCohort',
    relationship_types=['SIMILAR_JOURNEY'], node_labels=['Student'])
print(f'{louvain.community_count:,} journey cohorts, modularity {louvain.modularity:.4f}')

cohorts = as_frame(gds.graph.node_properties.stream(
    G, 'journeyCohort', node_labels=['Student']), 'journeyCohort')
ids = as_frame(gds.graph.node_properties.stream(
    G, 'id', node_labels=['Student']), 'studentId')
students = cohorts.merge(ids, on='nodeId')
students['studentId'] = students['studentId'].astype(int)
print('\nlargest cohorts:')
print(students['journeyCohort'].value_counts().head(10).to_string())

## 12. Do journey shapes track outcomes?

Same outcome path as the cohorts notebook:
`(:Student)-[:WAS_REGISTERED]->(:StudentRegistration)-[:CONTAINS_COURSE]->(:Course)`, with
`finalResult` on the last relationship.

The same caveat applies with more force here: a student with no interactions has no chain, no
embedding and no cohort, and that group is where the withdrawals concentrate.

In [ ]:
OUTCOME_QUERY = '''
MATCH (s:Student)-[:WAS_REGISTERED]->(:StudentRegistration)-[r:CONTAINS_COURSE]->(c:Course)
WHERE c.codeModule = $module
RETURN s.id AS studentId, r.finalResult AS finalResult
'''
with driver.session(database=NEO4J_DATABASE) as session:
    outcomes = (pd.DataFrame(session.run(OUTCOME_QUERY, module=MODULE).data())
                .drop_duplicates(subset='studentId'))

TOP_N = 8
biggest = students['journeyCohort'].value_counts().head(TOP_N).index
merged = students[students['journeyCohort'].isin(biggest)].merge(outcomes, on='studentId')

order = [c for c in ['Distinction', 'Pass', 'Fail', 'Withdrawn']
         if c in set(merged['finalResult'])]
share = (pd.crosstab(merged['journeyCohort'], merged['finalResult'],
                     normalize='index') * 100)[order].round(1)
share.insert(0, 'students', merged['journeyCohort'].value_counts()[share.index])
print(f'outcome mix by journey cohort, {TOP_N} biggest (% of each)')
print(share.to_string())

overall = (outcomes['finalResult'].value_counts(normalize=True) * 100).round(1)
print(f'\nwhole module {MODULE} for comparison:')
print(overall[order].to_string())
print(f"\nregistered students with no journey: "
      f"{outcomes['studentId'].nunique() - students['studentId'].nunique():,}")

ax = share[order].plot(kind='barh', stacked=True, figsize=(9, 0.5 * len(share) + 2))
ax.set_xlabel('% of cohort'); ax.set_ylabel('journey cohort')
ax.set_title(f'Outcomes by journey shape — module {MODULE}')
ax.legend(title='finalResult', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout(); plt.show()

## 13. What actually differs between the cohorts?

An embedding cohort is opaque on its own. These are the plainest summaries of the underlying
journeys — how long, how busy, when the activity happened, and how late it ran — which is
enough to see whether the grouping corresponds to anything a teacher would recognise.

In [ ]:
SHAPE_QUERY = '''
UNWIND $ids AS studentId
MATCH (s:Student {id: studentId})-[:FIRST_INTERACTION]->(:Interaction {module: $module})
MATCH (i:Interaction {module: $module, studentId: studentId})
RETURN studentId,
       count(i) AS events, sum(i.clicks) AS clicks,
       min(i.day) AS firstDay, max(i.day) AS lastDay,
       round(avg(i.day), 1) AS meanDay,
       count(DISTINCT i.activityTypeId) AS activityTypes
'''
with driver.session(database=NEO4J_DATABASE) as session:
    shapes = pd.DataFrame(session.run(
        SHAPE_QUERY, ids=students['studentId'].tolist(), module=MODULE).data())

profile = (students.merge(shapes, on='studentId')
           .groupby('journeyCohort')
           .agg(students=('studentId', 'size'), events=('events', 'mean'),
                clicks=('clicks', 'mean'), firstDay=('firstDay', 'mean'),
                lastDay=('lastDay', 'mean'), meanDay=('meanDay', 'mean'),
                activityTypes=('activityTypes', 'mean'))
           .round(1).sort_values('students', ascending=False))
print('journey shape by cohort')
print(profile.head(10).to_string())

## 14. Clean up

Two separate things to release, and **both matter**: the session is billed compute, and the
event chain is a few hundred thousand nodes sitting in your database.

`DELETE_CHAIN` defaults to `True` here, unlike the cohort property in the other notebook. A
property on existing nodes is cheap to leave behind; a quarter of a million extra nodes is
not. Set it to `False` if you want to keep the chain for another run — step 5 will skip
students that already have one.

In [ ]:
DELETE_CHAIN = True

try:
    G.drop(); print('projection dropped')
except Exception as error:
    print('projection:', error)
try:
    gds.delete(); print('session deleted')
except Exception as error:
    print('session:', error)

if DELETE_CHAIN:
    # Batched: DETACH DELETE over a few hundred thousand nodes in one
    # transaction is how you run a session out of memory.
    DELETE_QUERY = '''
    MATCH (i:Interaction {module: $module})
    WITH i LIMIT $batch
    DETACH DELETE i
    RETURN count(*) AS deleted
    '''
    removed = 0
    while True:
        with driver.session(database=NEO4J_DATABASE) as session:
            deleted = session.run(DELETE_QUERY, module=MODULE, batch=10000).single()['deleted']
        removed += deleted
        if deleted:
            print(f'  deleted {removed:,}', flush=True)
        if deleted == 0:
            break
    print(f'removed {removed:,} interaction nodes')
else:
    print('DELETE_CHAIN is False; the event chain is still in the database')

with driver.session(database=NEO4J_DATABASE) as session:
    left = session.run('MATCH (i:Interaction) RETURN count(i) AS n').single()['n']
    totals = session.run(
        'MATCH (n) WITH count(n) AS nodes '
        'MATCH ()-[r]->() RETURN nodes, count(r) AS relationships').single()
driver.close()

print(f'\ninteraction nodes remaining: {left:,}')
print(f"graph totals: {totals['nodes']:,} nodes, {totals['relationships']:,} relationships")
print('(66,920 nodes and 8,818,076 relationships is the untouched OULAD graph)')

## Where this differs from the cohorts notebook

Both group students and both compare against `finalResult`, but they ask different questions.
`aga_student_cohorts.ipynb` compares *sets* — which materials, weighted by clicks — so two
students who used the same pages score alike no matter when. FastPath compares *sequences*, so
a student who worked steadily and one who crammed the same material into the final fortnight
are different, and two students who did unrelated things at the same tempo can be alike.

## What the runs actually showed

Full module GGG, 2,359 students, twice — once without the event feature vector and once with
it, changing nothing else:

| | cohorts | modularity | outcome spread |
| --- | --- | --- | --- |
| no click intensity | 11 | 0.5470 | ~10 points |
| with click intensity | 10 | 0.5122 | ~14 points |

Adding intensity did **not** rescue the approach. Modularity fell slightly, the outcome spread
widened by a few points, and the shape table still shows every cohort landing within a narrow
band: mean day 104–119, first day 15–20, last day 198–216, 5.9–6.2 activity types. The one
cohort that stands out, 145, has both the highest click total (847 against a 470–665 range) and
the highest distinction rate (27.4% against 15.6% overall) — which is volume asserting itself,
not a distinctive trajectory.

For comparison, on this same data a single logged click total predicts pass or fail at 0.855
accuracy (see `aga_outcome_prediction.ipynb`), and set-overlap cohorts on module BBB separated
outcomes from 0.8% to 72.4% pass.

So the honest reading: **almost everyone who engaged with GGG at all spread their activity
across the whole presentation**, so there is little temporal variety for a sequence embedding to
find, and what signal exists is volume. That is a fact about this module, not about FastPath.
The algorithm would have more to work with where journeys genuinely diverge in shape — a module
with a hard mid-term deadline, or a dataset of customer journeys with distinct funnels. GGG has
the shortest chains of any OULAD module (average 119); `FFF` averages 478 and is the obvious
place to look next.